# Model Benchmark — Individual Performance & Resource Measurement (torch.profiler variant)

Sibling of `performance_and_resources_measurment.ipynb` — identical sections 0-7
and 9, but Section 8 uses `torch.profiler` instead of Scalene for the CPU-vs-device
time split, since it needs no subprocess (in-process context manager — no orphan-
process risk) and gives per-*operator* detail (which specific CUDA kernel dominates),
not just a percentage. Trade-off: it only sees genuinely PyTorch-dispatched ops, so
it's blind to `FaceDetector` (ONNXRuntime, not PyTorch) and has nothing meaningful
to say about `PersonTracker` with `with_reid=False` (no real torch ops — confirmed:
profiling it shows almost entirely profiler-induced sync overhead, not real cost).

Benchmarks every model in `lum_vision` **individually**, iterated over the real
images in `notebooks/input/` (one unrelated stock photo, `sandisk-...jpg`, is
excluded — it's a different size/aspect ratio and not representative office/camera
content), using:

- **`time.perf_counter_ns`** — wall-clock timing over multiple runs
- **`nvidia-smi`** — GPU memory in use (MB, steady-state footprint after warmup)
- **`psutil`** — process RSS (MB, steady-state footprint after warmup)

Every section produces two tables: the full per-image result set, and an average
across the image set per size bucket — plus detection counts (persons/faces found),
since the inputs are now real content instead of content-blind random noise.
Results are exported to `notebooks/output/benchmarks_detailed_torch.json` (every run) and
`notebooks/output/benchmarks_averaged_torch.json` (averaged per size bucket) — `_torch`
suffixed to avoid clobbering the Scalene sibling notebook's output in the same directory.

| Section | Model | Input varied |
|---|---|---|
| 1 | `PersonDetector` | resolution (resized real images) + native size, × 7 images |
| 2 | `FaceDetector` | resolution (resized real images) + native size, × 7 images |
| 3 | `PersonTracker` | resolution + native, real detections, × 7 images |
| 4 | `FaceMatcher` | gallery size, real query embedding per image |
| 5 | `GlobalTrackManager` | crop size + native, real person crops, × 7 images |
| 6 | `ActionRecognizer` | fixed crop size, real person crop per image |
| 7 | Isolated footprint | one fresh subprocess per model, nothing else loaded |
| 8 | `torch.profiler` CPU-vs-device profile | per-op CPU/CUDA time split + top kernel, torch-based models only |

**A note on `cpu_rss_mb`/`gpu_mb` in sections 1-6:** every model is loaded into the
same shared process (see "Initialize models" below), so those numbers are
cumulative — a snapshot of the whole process at that point, not this model alone —
and `gpu_mb` itself is device-wide (every process on the machine, not just this
one). Section 7 fixes this with isolated, per-model-only measurements.

## 0. Setup

In [ ]:
import gc, json, os, subprocess, sys, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import psutil
import torch

import lum_vision
from lum_vision import (
    ActionConfig,
    ActionRecognizer,
    FaceDetector,
    FaceMatcher,
    InMemoryEmbeddingProvider,
    ModelFactory,
    PersonTracker,
    VisionConfig,
)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = REPO / "notebooks" / "output"
OUT.mkdir(parents=True, exist_ok=True)

print(f"lum_vision {lum_vision.__version__}")
print(f"python    {sys.executable}")
print(f"torch     {torch.__version__} | cuda: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU       {torch.cuda.get_device_name(0)}")
    print(f"VRAM      {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"psutil    : {psutil.__version__}")

In [ ]:
config = VisionConfig()

print(f"model_cache_dir : {config.model_cache_dir}")
print(f"  weights       : {config.weights_dir}")
print(f"  insightface   : {config.insightface_dir}")

### Input images

Real images from `notebooks/input/`, excluding the unrelated stock photo. Kept at
native resolution here — individual sections resize as needed.

In [ ]:
INPUT_DIR = REPO / "notebooks" / "input"
EXCLUDE = {"sandisk-pAKgXLu04CQ-unsplash.jpg"}  # unrelated stock photo, different size/content

image_paths = sorted(
    p for p in INPUT_DIR.glob("*")
    if p.name not in EXCLUDE and p.suffix.lower() in {".png", ".jpg", ".jpeg"}
)
if not image_paths:
    raise FileNotFoundError(f"No usable images found in {INPUT_DIR}")

IMAGES = []
for p in image_paths:
    img = cv2.imread(str(p))
    if img is None:
        raise RuntimeError(f"Could not decode {p}")
    IMAGES.append((p.name, img))

print(f"Loaded {len(IMAGES)} real images from {INPUT_DIR}:")
for name, img in IMAGES:
    print(f"  {name:35s} {img.shape[1]}x{img.shape[0]}")

### Utility functions

In [ ]:
def get_gpu_mem_mb():
    """Return current GPU memory used in MB, or None if unavailable."""
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5,
        )
        return float(r.stdout.strip().split("\n")[0])
    except Exception:
        return None


def resize_to(img, w, h):
    """Resize a real image to an exact (w, h), no aspect-ratio preservation."""
    return cv2.resize(img, (w, h))


def crop_box(img, bbox, pad=0):
    """Crop a bbox out of img, clamped to image bounds."""
    h, w = img.shape[:2]
    x1, y1, x2, y2 = (int(v) for v in bbox[:4])
    x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
    x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
    return img[y1:y2, x1:x2]


def scale_dets(persons, orig_shape, new_shape):
    """Rescale detection bboxes from one frame size to another (e.g. after resize_to)."""
    oh, ow = orig_shape[:2]
    nh, nw = new_shape[:2]
    sx, sy = nw / ow, nh / oh
    scaled = []
    for p in persons:
        x1, y1, x2, y2 = p["bbox"][:4]
        scaled.append({"bbox": [x1 * sx, y1 * sy, x2 * sx, y2 * sy], "confidence": p["confidence"], "keypoints": None})
    return scaled


def bench(fn, *, warmup=2, repeats=10, label="", capture_last=False):
    """Benchmark a zero-argument callable.

    Measures wall-clock time (mean +/- std) over repeats runs, plus resource
    usage as plain resident numbers, not before/after deltas: warmup already
    brings memory to steady state (allocator pools, loaded weights), so a
    delta across repeated same-shape calls is almost always ~0 — more
    confusing than useful. cpu_rss_mb and gpu_mb are "how much is actually
    in use", sampled once warmup has settled. tracemalloc_peak_mb is kept
    separately since it genuinely varies (Python-level allocation peak
    during the timed calls, e.g. grows with input resolution).

    If capture_last is True, the return value of the final timed call is
    stashed under "_raw" (pop it before exporting — it isn't JSON-safe).
    """
    import tracemalloc
    process = psutil.Process(os.getpid())

    # --- warmup: let allocators / caches reach steady state before measuring ---
    for _ in range(warmup):
        fn()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    cpu_rss_mb = process.memory_info().rss / 1024**2
    gpu_mb = get_gpu_mem_mb()

    # --- timed runs ---
    tracemalloc.start()
    times_ns = []
    last_out = None
    for _ in range(repeats):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter_ns()
        out = fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter_ns()
        times_ns.append(t1 - t0)
        if capture_last:
            last_out = out
    _current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    times_ms = [t / 1e6 for t in times_ns]
    result = {
        "label": label,
        "wall_ms_mean": round(float(np.mean(times_ms)), 2),
        "wall_ms_std": round(float(np.std(times_ms)), 2),
        "cpu_rss_mb": round(cpu_rss_mb, 1),
        "gpu_mb": gpu_mb,
        "tracemalloc_peak_mb": round(peak / 1024**2, 2),
        "repeats": repeats,
    }
    if capture_last:
        result["_raw"] = last_out
    gpu_str = f'{result["gpu_mb"]:.0f} MB' if result["gpu_mb"] is not None else "N/A"
    print(
        "  " + label.ljust(40)
        + "  wall=" + f'{result["wall_ms_mean"]:8.2f}'
        + " \u00b1 " + f'{result["wall_ms_std"]:.2f}' + " ms"
        + "  | RAM=" + f'{result["cpu_rss_mb"]:.0f}' + " MB"
        + "  | GPU=" + gpu_str
    )
    return result


def run_benchmark_over_images(call_fn, size_buckets, images, model_name, count_fn=None, repeats=10, warmup=2):
    """Benchmark call_fn(frame) across every (size_bucket, image) pair.

    call_fn(frame) -> zero-arg callable to bench().
    count_fn(raw_result) -> int, optional (needs capture_last, handled here).
    Returns (rows, per_run_df, averaged_df) — averaged_df is the mean per size_bucket
    across the image set.
    """
    rows = []
    for label, size in size_buckets:
        for img_name, img in images:
            frame = resize_to(img, *size) if size else img
            r = bench(call_fn(frame), label=f"{label} | {img_name}", repeats=repeats, warmup=warmup,
                      capture_last=count_fn is not None)
            raw = r.pop("_raw", None)
            r["model"] = model_name
            r["size_bucket"] = label
            r["image"] = img_name
            r["resolution"] = f"{frame.shape[1]}x{frame.shape[0]}"
            if count_fn:
                r["count"] = count_fn(raw)
            rows.append(r)
    df = pd.DataFrame(rows)
    agg_cols = ["wall_ms_mean", "cpu_rss_mb", "tracemalloc_peak_mb", "gpu_mb"]
    if count_fn:
        agg_cols.append("count")
    avg = df.groupby("size_bucket", sort=False)[agg_cols].mean().round(2).reset_index()
    return rows, df, avg


def measure_footprint(label, code_str, timeout=90):
    """Measure a model's GPU/RAM footprint in complete isolation.

    Runs code_str (imports + model construction + one warm call) in a
    brand-new subprocess, snapshotting GPU/RAM immediately before and after
    inside that same process. This is isolated from every other model this
    notebook has already loaded (a fresh process starts with nothing of
    ours loaded) and from anything else already running on this machine's
    GPU (both snapshots include that same pre-existing baseline, so it
    cancels out of the delta either way).
    """
    preamble = '''
import subprocess, psutil, os, json as _json

def _gpu_mb():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                            capture_output=True, text=True, timeout=5)
        return float(r.stdout.strip().splitlines()[0])
    except Exception:
        return None

_proc = psutil.Process(os.getpid())
_before = {"gpu_mb": _gpu_mb(), "cpu_rss_mb": _proc.memory_info().rss / 1024**2}
'''
    postamble = '''
_after = {"gpu_mb": _gpu_mb(), "cpu_rss_mb": _proc.memory_info().rss / 1024**2}
print("FOOTPRINT_JSON=" + _json.dumps({"before": _before, "after": _after}))
'''
    script = preamble + code_str + postamble
    try:
        r = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True, timeout=timeout)
    except subprocess.TimeoutExpired:
        print(f"  {label}: timed out")
        return {"model": label, "gpu_mb": None, "cpu_rss_mb": None}

    for line in r.stdout.splitlines():
        if line.startswith("FOOTPRINT_JSON="):
            data = json.loads(line[len("FOOTPRINT_JSON="):])
            before, after = data["before"], data["after"]
            gpu_delta = None
            if before["gpu_mb"] is not None and after["gpu_mb"] is not None:
                gpu_delta = round(after["gpu_mb"] - before["gpu_mb"], 1)
            cpu_delta = round(after["cpu_rss_mb"] - before["cpu_rss_mb"], 1)
            print(f"  {label:28s}  GPU={gpu_delta} MB  RAM={cpu_delta} MB")
            return {"model": label, "gpu_mb": gpu_delta, "cpu_rss_mb": cpu_delta}

    print(f"  {label}: measurement failed")
    print(r.stderr[-500:])
    return {"model": label, "gpu_mb": None, "cpu_rss_mb": None}


def torch_profile_op(label, fn, warmup=2, repeats=20):
    """Profile a zero-arg callable with torch.profiler: CPU-vs-device time split.

    Needs no subprocess (unlike Scalene) — torch.profiler instruments PyTorch's
    own op dispatcher directly via a context manager in this process, so there's
    no cold-start-contamination problem to work around: warmup runs outside the
    `with` block, only the timed calls go inside it. No orphan-process risk either.

    Only meaningful for genuinely PyTorch-dispatched models — it can't see into
    ONNXRuntime's own execution engine (FaceDetector), and has nothing to say
    about ops that never touch a torch tensor (PersonTracker with with_reid=False
    is pure Python/NumPy association logic, confirmed by testing: profiling it
    shows almost entirely profiler-induced sync overhead, not real op cost).

    The naive approach — summing self_device_time_total across every event in
    key_averages() — double-counts: PyTorch's profiler represents the same GPU
    work twice (once on the CPU-side dispatch event like aten::cudnn_convolution,
    again on the underlying CUDA kernel event). The filter below is copied from
    torch's own internal EventList.table() printing logic (profiler_util.py) —
    confirmed it exactly reproduces the "Self CPU/CUDA time total" table footer.
    """
    from torch.autograd import DeviceType
    from torch.profiler import ProfilerActivity, profile

    for _ in range(warmup):
        fn()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    with profile(activities=activities) as prof:
        for _ in range(repeats):
            fn()
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    events = prof.key_averages()
    cpu_total = sum(e.self_cpu_time_total for e in events)
    device_total = 0.0
    device_events = []
    for e in events:
        if e.device_type == DeviceType.CPU and e.is_legacy:
            device_total += e.self_device_time_total
        elif e.device_type in (DeviceType.CUDA, DeviceType.PrivateUse1, DeviceType.MTIA, DeviceType.XPU) \
                and not e.is_user_annotation:
            device_total += e.self_device_time_total
            device_events.append(e)

    total = cpu_total + device_total
    cpu_pct = 100 * cpu_total / total if total else 0.0
    device_pct = 100 * device_total / total if total else 0.0
    top = max(device_events, key=lambda e: e.self_device_time_total, default=None)
    top_kernel = top.key if top is not None else "N/A"

    print(
        f"  {label:28s}  cpu={cpu_pct:5.1f}%  device={device_pct:5.1f}%"
        f"  (cpu={cpu_total / 1000 / repeats:.3f}ms/call  device={device_total / 1000 / repeats:.3f}ms/call)"
        f"  top_kernel={top_kernel}"
    )
    return {
        "model": label,
        "cpu_percent": round(cpu_pct, 2),
        "device_percent": round(device_pct, 2),
        "cpu_ms_per_call": round(cpu_total / 1000 / repeats, 4),
        "device_ms_per_call": round(device_total / 1000 / repeats, 4),
        "top_kernel": top_kernel,
    }


SIZE_BUCKETS = [
    ("320x240", (320, 240)),
    ("640x480", (640, 480)),
    ("1280x720", (1280, 720)),
    ("1920x1080", (1920, 1080)),
    ("native", None),
]

### Initialize models (lazy — nothing loads until first access)

In [ ]:
models = ModelFactory(config)

# Eagerly trigger model loading (weights download on first run)
_ = models.person_detector
_ = models.face_detector
_ = models.global_track_manager

print("All detection models loaded.")
print(f"  PersonDetector : {models.person_detector.model.model_name}")
print(f"  FaceDetector   : {models.face_detector.model_name}")
print(f"  GlobalTrack    : {models.global_track_manager.reid_model_name}")

### Precompute native-resolution real detections

Sections 3-6 need real persons/faces to work with (tracking, matching, ReID,
action) — detect once per image at native resolution here and reuse below,
instead of paying detection cost again in every downstream section.

In [ ]:
detector = models.person_detector
face_det = models.face_detector

NATIVE = {}
print("Native-resolution detection pass:")
for img_name, img in IMAGES:
    r = detector.model(img, verbose=False)[0]
    persons = [
        {"bbox": b, "confidence": float(c), "keypoints": None}
        for b, c, k in zip(r.boxes.xyxy.tolist(), r.boxes.conf.tolist(), r.boxes.cls.tolist())
        if int(k) == 0 and c >= config.person_detection_threshold
    ]
    faces = face_det.extract_face_features(img)
    NATIVE[img_name] = {"image": img, "persons": persons, "faces": faces}
    print(f"  {img_name:35s} persons={len(persons):2d}  faces={len(faces)}")

---

## 1. PersonDetector (YOLO26s)

Measures YOLO inference time and memory across resolutions and the native size,
iterated over every real image. (`cpu_rss_mb`/`gpu_mb` are whole-process
snapshots, not PersonDetector alone — see Section 7 for an isolated number.)

In [ ]:
def count_persons(result):
    r = result[0]
    return sum(
        1 for c, k in zip(r.boxes.conf.tolist(), r.boxes.cls.tolist())
        if int(k) == 0 and c >= config.person_detection_threshold
    )

person_rows, person_df, person_avg = run_benchmark_over_images(
    call_fn=lambda frame: (lambda: detector.model(frame, verbose=False)),
    size_buckets=SIZE_BUCKETS,
    images=IMAGES,
    model_name="PersonDetector",
    count_fn=count_persons,
)

print("\nPersonDetector — per image x size:")
display(person_df[["size_bucket", "image", "resolution", "wall_ms_mean", "wall_ms_std",
                    "tracemalloc_peak_mb", "gpu_mb", "count"]])
print("\nPersonDetector — averaged across images per size:")
display(person_avg)

---

## 2. FaceDetector (InsightFace buffalo_l)

Includes face detection + 512-d embedding extraction, across resolutions and the
native size, iterated over every real image. (`cpu_rss_mb`/`gpu_mb` are
whole-process snapshots, not FaceDetector alone — see Section 7 for an isolated
number.)

In [ ]:
def count_faces(result):
    return len(result)

face_rows, face_df, face_avg = run_benchmark_over_images(
    call_fn=lambda frame: (lambda: face_det.extract_face_features(frame)),
    size_buckets=SIZE_BUCKETS,
    images=IMAGES,
    model_name="FaceDetector",
    count_fn=count_faces,
)

print("\nFaceDetector — per image x size:")
display(face_df[["size_bucket", "image", "resolution", "wall_ms_mean", "wall_ms_std",
                  "tracemalloc_peak_mb", "gpu_mb", "count"]])
print("\nFaceDetector — averaged across images per size:")
display(face_avg)

### FaceDetector — person crop vs full frame

Section 2 above always feeds FaceDetector the full frame (that's the right input
for scanning an unknown scene). Here's the alternative: crop to the first detected
person (from the precompute pass) and run FaceDetector on *just that crop* instead
— smaller input, less background, but only searches within one person's box rather
than the whole scene. Compared at each crop's own native size (no resize sweep —
crops are already much smaller than the full frame, so sweeping the same
320x240-1920x1080 buckets would mean upscaling most of them).

In [ ]:
face_crop_rows = []
for img_name, entry in NATIVE.items():
    persons = entry["persons"]
    if not persons:
        print(f"  skip {img_name}: no persons detected")
        continue
    crop = crop_box(entry["image"], persons[0]["bbox"])
    if crop.size == 0:
        print(f"  skip {img_name}: empty crop")
        continue

    r = bench(lambda c=crop: face_det.extract_face_features(c), label=img_name, capture_last=True)
    faces = r.pop("_raw")
    r["model"] = "FaceDetector_on_crop"
    r["image"] = img_name
    r["resolution"] = f"{crop.shape[1]}x{crop.shape[0]}"
    r["count"] = len(faces)
    face_crop_rows.append(r)

face_crop_df = pd.DataFrame(face_crop_rows)
print("\nFaceDetector on person crop (native crop size) — per image:")
display(face_crop_df[["image", "resolution", "wall_ms_mean", "wall_ms_std",
                       "tracemalloc_peak_mb", "gpu_mb", "count"]])

face_native = face_avg[face_avg["size_bucket"] == "native"].iloc[0]
comparison = pd.DataFrame([
    {"input": "full_frame (native)", "wall_ms_mean": face_native["wall_ms_mean"],
     "count": face_native["count"], "gpu_mb": face_native["gpu_mb"], "cpu_rss_mb": face_native["cpu_rss_mb"]},
    {"input": "person_crop (native)", "wall_ms_mean": round(face_crop_df["wall_ms_mean"].mean(), 2),
     "count": round(face_crop_df["count"].mean(), 2), "gpu_mb": round(face_crop_df["gpu_mb"].mean(), 1),
     "cpu_rss_mb": round(face_crop_df["cpu_rss_mb"].mean(), 1)},
])
print("\nFull-frame vs person-crop — averaged across images:")
display(comparison)

---

## 3. PersonTracker (BoT-SORT)

Benchmarks the tracking `update()` step alone (detection cost excluded — reuses
the real detections from the precompute pass above, rescaled to each size bucket).

**Caveat:** each real image is a single still, not a video clip, so there's no real
motion to track. A fresh tracker is built per (size, image) pair and fed the *same*
real detections for a handful of repeated `update()` calls — this measures
steady-state per-call cost, not actual cross-frame ID persistence. 320x240 is
skipped here (too small for meaningful person boxes, matching the original sweep).
Images with zero detected persons are skipped.

Raises a `RuntimeError` if you have upstream boxmot rather than the fork — that is
deliberate, because the fallback path silently tracks worse.

(`cpu_rss_mb`/`gpu_mb` are whole-process snapshots, not PersonTracker alone —
see Section 7 for an isolated number.)

In [ ]:
TRACK_BUCKETS = [b for b in SIZE_BUCKETS if b[0] != "320x240"]
TRACK_REPEATS = 5

tracker_rows = []
for label, size in TRACK_BUCKETS:
    for img_name, entry in NATIVE.items():
        img = entry["image"]
        frame = resize_to(img, *size) if size else img
        dets = scale_dets(entry["persons"], img.shape, frame.shape) if size else entry["persons"]
        if not dets:
            print(f"  skip {label} | {img_name}: no persons detected")
            continue

        tracker = PersonTracker(camera_id=0, with_reid=False, confidence_threshold=0.5)

        def do_update(d=dets, f=frame, t=tracker):
            return t.update(d, frame=f)

        r = bench(do_update, repeats=TRACK_REPEATS, warmup=1, label=f"{label} | {img_name}", capture_last=True)
        active, _removed = r.pop("_raw")
        r["model"] = "PersonTracker"
        r["size_bucket"] = label
        r["image"] = img_name
        r["resolution"] = f"{frame.shape[1]}x{frame.shape[0]}"
        r["count"] = len(active)
        tracker_rows.append(r)

tracker_df = pd.DataFrame(tracker_rows)
tracker_avg = tracker_df.groupby("size_bucket", sort=False)[
    ["wall_ms_mean", "cpu_rss_mb", "tracemalloc_peak_mb", "gpu_mb", "count"]
].mean().round(2).reset_index()

print("\nPersonTracker — per image x size:")
display(tracker_df[["size_bucket", "image", "resolution", "wall_ms_mean", "wall_ms_std",
                     "tracemalloc_peak_mb", "gpu_mb", "count"]])
print("\nPersonTracker — averaged across images per size:")
display(tracker_avg)

---

## 4. FaceMatcher

Benchmarks cosine similarity computation across different gallery sizes (pure NumPy
on CPU) — gallery size is a database-scale axis, unrelated to image resolution, so
it stays synthetic. The **query** embedding is now real: the first detected face per
image. Images with zero detected faces are skipped.

In [ ]:
GALLERY_SIZES = [10, 100, 1000, 5000]
EMBED_DIM = 512

matcher_rows = []
for img_name, entry in NATIVE.items():
    faces = entry["faces"]
    if not faces:
        print(f"  skip {img_name}: no faces detected")
        continue
    query = faces[0]["embedding"].reshape(1, -1).astype(np.float32)

    for n in GALLERY_SIZES:
        names = [f"person_{i}" for i in range(n)]
        embs = np.random.randn(n, EMBED_DIM).astype(np.float32)
        embs /= np.linalg.norm(embs, axis=1, keepdims=True)
        provider = InMemoryEmbeddingProvider(names, embs)
        matcher = FaceMatcher(provider=provider, match_threshold=0.3)

        r = bench(lambda q=query, m=matcher: m.compute_similarities(q), label=f"gallery={n} | {img_name}")
        r["model"] = "FaceMatcher"
        r["size_bucket"] = f"gallery={n}"
        r["image"] = img_name
        r["gallery_size"] = n
        matcher_rows.append(r)

matcher_df = pd.DataFrame(matcher_rows)
matcher_avg = matcher_df.groupby("size_bucket", sort=False)[
    ["wall_ms_mean", "cpu_rss_mb", "tracemalloc_peak_mb"]
].mean().round(3).reset_index()

print("\nFaceMatcher — per image x gallery size:")
display(matcher_df[["size_bucket", "image", "wall_ms_mean", "wall_ms_std",
                     "tracemalloc_peak_mb", "cpu_rss_mb"]])
print("\nFaceMatcher — averaged across images per gallery size:")
display(matcher_avg)

---

## 5. GlobalTrackManager — ReID embedding extraction

Measures the time and memory to extract a body ReID embedding (OSNet) from a real
person crop (the first detected person per image), resized to each crop-size bucket
plus its native crop size. Images with zero detected persons are skipped.

(`cpu_rss_mb`/`gpu_mb` are whole-process snapshots, not this model alone — see
Section 7 for an isolated number.)

In [ ]:
CROP_BUCKETS = [("64x128", (64, 128)), ("128x256", (128, 256)), ("256x512", (256, 512)), ("native", None)]
gtm = models.global_track_manager

reid_rows = []
for img_name, entry in NATIVE.items():
    persons = entry["persons"]
    if not persons:
        print(f"  skip {img_name}: no persons detected")
        continue
    crop = crop_box(entry["image"], persons[0]["bbox"])
    if crop.size == 0:
        print(f"  skip {img_name}: empty crop")
        continue

    for label, size in CROP_BUCKETS:
        c = cv2.resize(crop, size) if size else crop
        r = bench(lambda c=c: gtm._extract_body_embedding(c), label=f"{label} | {img_name}")
        r["model"] = "GlobalTrackManager_ReID"
        r["size_bucket"] = label
        r["image"] = img_name
        r["resolution"] = f"{c.shape[1]}x{c.shape[0]}"
        reid_rows.append(r)

reid_df = pd.DataFrame(reid_rows)
reid_avg = reid_df.groupby("size_bucket", sort=False)[
    ["wall_ms_mean", "cpu_rss_mb", "tracemalloc_peak_mb", "gpu_mb"]
].mean().round(2).reset_index()

print("\nGlobalTrackManager ReID — per image x crop size:")
display(reid_df[["size_bucket", "image", "resolution", "wall_ms_mean", "wall_ms_std",
                  "tracemalloc_peak_mb", "gpu_mb"]])
print("\nGlobalTrackManager ReID — averaged across images per crop size:")
display(reid_avg)

---

## 6. ActionRecognizer (Ollama VLM)

**Note:** Timings include network latency to the Ollama server. If Ollama is
unreachable, the whole section is skipped per-image. Uses a real person crop (the
first detected person per image), resized to a fixed crop size — this stage is
network-bound, not size-bound, so no size sweep here.

In [ ]:
ACTION_CROP_SIZE = (256, 512)

action_config = ActionConfig(
    enabled=True,
    ollama_api_url="http://localhost:11534",
    model_name="gemma3:4b",
    inference_timeout=60,
    actions={
        "sleeping":              {"backend_type": "sleeping",    "description": "head resting on desk"},
        "using phone":           {"backend_type": "phone_usage", "description": "holding a phone"},
        "working with computer": {"backend_type": "working",     "description": "sitting at a desk"},
        "talking with someone":  {"backend_type": "talking",     "description": "facing another person"},
        "idle":                  {"backend_type": "unknown",     "description": "no clear activity"},
    },
)

recognizer = ActionRecognizer(action_config)
print(f"Ollama URL : {action_config.ollama_api_url}")
print(f"Model      : {action_config.model_name}")
print(f"Crop size  : {ACTION_CROP_SIZE[0]}x{ACTION_CROP_SIZE[1]}")
print("NOTE       : timings include Ollama network round-trip\n")

action_rows = []
for img_name, entry in NATIVE.items():
    persons = entry["persons"]
    if not persons:
        print(f"  skip {img_name}: no persons detected")
        continue
    crop = crop_box(entry["image"], persons[0]["bbox"])
    if crop.size == 0:
        print(f"  skip {img_name}: empty crop")
        continue
    crop = cv2.resize(crop, ACTION_CROP_SIZE)

    try:
        first = recognizer.recognize(crop, metadata={"track_id": 0})
    except Exception as e:
        print(f"  SKIPPED (all images): {e}")
        print("  Start Ollama with: docker compose up ollama")
        break

    if first is None:
        print(f"  {img_name}: inference returned None — Ollama may be unreachable")
        continue

    r = bench(lambda c=crop: recognizer.recognize(c, metadata={"track_id": 0}), warmup=0, repeats=3, label=img_name)
    r["model"] = "ActionRecognizer"
    r["image"] = img_name
    r["action"] = first.action
    action_rows.append(r)

if action_rows:
    action_df = pd.DataFrame(action_rows)
    print("\nActionRecognizer — per image:")
    display(action_df[["image", "action", "wall_ms_mean", "wall_ms_std"]])
else:
    action_df = pd.DataFrame()
    print("\nNo ActionRecognizer results (Ollama unreachable or no person crops).")

---

## 7. Isolated per-model resource footprint

Each model here is loaded **alone**, in a brand-new subprocess, with nothing else
of ours resident — unlike sections 1-6, where every model shares one process and
`gpu_mb` is device-wide (nvidia-smi sees every process on the machine, not just
this one). GPU/RAM are snapshotted immediately before and after inside that same
subprocess, so the delta is isolated from every other model in this notebook *and*
from anything else already running on the GPU — both snapshots include that same
baseline, so it cancels out of the difference either way.

`FaceMatcher` and `ActionRecognizer` are excluded: neither loads local model
weights (`FaceMatcher` is pure NumPy over an in-memory array; `ActionRecognizer`
calls Ollama over the network), so there's no local footprint to isolate.

**Shared import tax:** `lum_vision/__init__.py` eagerly imports every submodule —
InsightFace, ONNXRuntime, boxmot, Ollama's client, etc. — regardless of which
model you actually construct. So `import lum_vision` alone, before touching any
model, already costs real memory. The `baseline` row below measures exactly that
(import + an empty `ModelFactory()`, nothing constructed) — every model's own row
includes this same baseline on top of its own marginal cost, so compare each
model against `baseline`, not against zero.

In [ ]:
FOOTPRINT_MODELS = [
    ("baseline (import + factory only)", '''
import lum_vision
from lum_vision import VisionConfig, ModelFactory
config = VisionConfig()
models = ModelFactory(config)
'''),
    ("PersonDetector", '''
import numpy as np
from lum_vision import VisionConfig, ModelFactory
config = VisionConfig()
models = ModelFactory(config)
_ = models.person_detector
frame = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)
models.person_detector.model(frame, verbose=False)
'''),
    ("FaceDetector", '''
import numpy as np
from lum_vision import VisionConfig, ModelFactory
config = VisionConfig()
models = ModelFactory(config)
_ = models.face_detector
frame = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)
models.face_detector.extract_face_features(frame)
'''),
    ("GlobalTrackManager_ReID", '''
import numpy as np
from lum_vision import VisionConfig, ModelFactory
config = VisionConfig()
models = ModelFactory(config)
gtm = models.global_track_manager
crop = np.random.randint(0, 255, (256, 128, 3), dtype=np.uint8)
gtm._extract_body_embedding(crop)
'''),
    ("PersonTracker", '''
import numpy as np
from lum_vision import PersonTracker
tracker = PersonTracker(camera_id=0, with_reid=False, confidence_threshold=0.5)
dets = [{"bbox": [100, 100, 300, 400], "confidence": 0.8, "keypoints": None}]
frame = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)
tracker.update(dets, frame=frame)
'''),
]

print("Isolated per-model footprint (fresh subprocess each, nothing else loaded):")
footprint_rows = [measure_footprint(name, code_str) for name, code_str in FOOTPRINT_MODELS]
footprint_df = pd.DataFrame(footprint_rows)
print()
display(footprint_df)

---

## 8. `torch.profiler` CPU-vs-device profile

Where each model's measured time actually goes: CPU-side dispatch, or the CUDA
device it launches kernels on — plus, unlike a generic profiler, exactly *which*
kernel dominates. No subprocess needed: `torch.profiler` is a context manager
that instruments PyTorch's own op dispatcher directly in this process, so warmup
just runs outside the `with` block — no cold-start-contamination problem to work
around, and no orphan-process risk the way Scalene's subprocess-per-profile
approach had.

**Trade-off:** it only sees ops genuinely dispatched through PyTorch, so it's
blind to `FaceDetector` (ONNXRuntime, not PyTorch — a different execution engine
entirely) and has nothing meaningful to say about `PersonTracker` with
`with_reid=False` — confirmed by testing: profiling it shows almost entirely
profiler-induced sync overhead (`cudaDeviceSynchronize`), not real op cost, since
it never dispatches a real torch tensor op. `FaceMatcher` (pure NumPy) and
`ActionRecognizer` (network-bound) are excluded for the same reason as before.

Reuses the already-loaded `detector`/`gtm` instances and the real representative
image/crop directly in memory — no subprocess boundary to cross, so no need to
write anything to disk first.

In [ ]:
rep_img_name, rep_entry = next((n, e) for n, e in NATIVE.items() if e["persons"])
rep_frame = rep_entry["image"]
rep_crop = crop_box(rep_entry["image"], rep_entry["persons"][0]["bbox"])
print(f"Representative image: {rep_img_name}  ({len(rep_entry['persons'])} persons)")

torch_profile_rows = [
    torch_profile_op("PersonDetector", lambda: detector.model(rep_frame, verbose=False), repeats=20),
    torch_profile_op("GlobalTrackManager_ReID", lambda: gtm._extract_body_embedding(rep_crop), repeats=20),
]
torch_profile_df = pd.DataFrame(torch_profile_rows)
print()
display(torch_profile_df)

---

## 9. Summary & export

In [ ]:
# Detailed (every run) table across all six models
detailed_frames = [person_df, face_df, tracker_df, matcher_df, reid_df]
if not action_df.empty:
    detailed_frames.append(action_df)
detailed_df = pd.concat(detailed_frames, ignore_index=True, sort=False)

# Averaged-across-images table per model x size bucket
averaged_frames = [
    person_avg.assign(model="PersonDetector"),
    face_avg.assign(model="FaceDetector"),
    tracker_avg.assign(model="PersonTracker"),
    matcher_avg.assign(model="FaceMatcher"),
    reid_avg.assign(model="GlobalTrackManager_ReID"),
]
averaged_df = pd.concat(averaged_frames, ignore_index=True, sort=False)

print("Detailed results — every (model, size, image) run:")
display(detailed_df)
print()
print("Averaged results — mean across images per (model, size):")
display(averaged_df)
print()
print("Isolated per-model footprint (Section 7):")
display(footprint_df)
print()
print("torch.profiler CPU-vs-device profile (Section 8):")
display(torch_profile_df)

# Export: full detail + averaged summary + isolated footprint + torch profile
# Distinct filenames from performance_and_resources_measurment.ipynb (the Scalene
# sibling) — both notebooks share notebooks/output/, so same names would clobber.
detailed_path = OUT / "benchmarks_detailed_torch.json"
detailed_df.to_json(detailed_path, orient="records", indent=2)
print(f"\nSaved {len(detailed_df)} detailed rows to {detailed_path}")

averaged_path = OUT / "benchmarks_averaged_torch.json"
averaged_df.to_json(averaged_path, orient="records", indent=2)
print(f"Saved {len(averaged_df)} averaged rows to {averaged_path}")

# Full export: system info + both tables
system_info = {
    "lum_vision_version": lum_vision.__version__,
    "python": sys.executable,
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_total_mb": round(torch.cuda.get_device_properties(0).total_memory / 1024**2) if torch.cuda.is_available() else None,
    "cpu_ram_gb": round(psutil.virtual_memory().total / 1024**3, 1),
    "images_used": [name for name, _ in IMAGES],
}

export = {
    "system_info": system_info,
    "detailed": detailed_df.to_dict(orient="records"),
    "averaged": averaged_df.to_dict(orient="records"),
    "isolated_footprint": footprint_df.to_dict(orient="records"),
    "torch_profile": torch_profile_df.to_dict(orient="records"),
}
full_json = OUT / "benchmarks_full_torch.json"
with open(full_json, "w") as f:
    json.dump(export, f, indent=2, default=str)

print(f"Saved full results to {full_json}")
print()
print("System info:")
for k, v in system_info.items():
    print(f"  {k:20s}: {v}")